In [0]:
%sql
SELECT * FROM shopstream.core.silver_orders limit 10

order_line_id,order_id,customer_id,product_id,quantity,unit_price,line_revenue,order_ts,status,coupon_code
L0000001,O000001,C00628,P0123,1,106.28,106.28,2026-02-20T07:14:43.000Z,completed,FESTIVE20
L0000002,O000001,C00628,P0096,1,267.14,267.14,2026-02-20T07:14:43.000Z,completed,WELCOME15
L0000003,O000002,C00141,P0057,2,22.09,44.18,2026-02-07T07:02:26.000Z,completed,null
L0000004,O000002,C00141,P0017,1,53.8,53.8,2026-02-07T07:02:26.000Z,completed,FESTIVE20
L0000005,O000002,C00141,P0126,1,107.67,107.67,2026-02-07T07:02:26.000Z,completed,FESTIVE20
L0000006,O000002,C00141,P0110,1,27.62,27.62,2026-02-07T07:02:26.000Z,completed,null
L0000008,O000004,C00463,P0011,1,64.02,64.02,2026-04-27T01:56:15.000Z,returned,null
L0000009,O000004,C00463,P0113,1,319.88,319.88,2026-04-27T01:56:15.000Z,returned,null
L0000010,O000004,C00463,P0074,1,156.41,156.41,2026-04-27T01:56:15.000Z,returned,WELCOME15
L0000011,O000004,C00463,P0014,1,122.93,122.93,2026-04-27T01:56:15.000Z,returned,null


##1. Daily revenue 

In [0]:
%sql
CREATE OR REPLACE TABLE shopstream.core.gold_orders AS 
SELECT 
    DATE(order_ts)              AS order_date,
    COUNT(DISTINCT order_id)    AS orders,
    SUM(quantity)               AS units_sold,
    ROUND(SUM(line_revenue),2)  AS revenue
FROM shopstream.core.silver_orders
WHERE status = 'completed'
GROUP BY DATE(order_ts);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM shopstream.core.gold_orders LIMIT 10

order_date,orders,units_sold,revenue
2026-06-08,25,53,9151.06
2026-02-28,31,65,9103.39
2026-06-11,34,79,11904.83
2026-02-13,31,67,10448.42
2026-05-05,33,64,9215.92
2026-05-13,27,69,12621.02
2026-06-28,34,81,13279.05
2026-01-24,38,83,13804.05
2026-05-16,39,91,15951.77
2026-02-09,23,54,8742.54


## 2. Category performance

In [0]:
%sql
CREATE OR REPLACE TABLE shopstream.core.gold_category_performance AS 
SELECT 
    p.category,
    COUNT(DISTINCT o.order_id)          AS orders,
    SUM(o.quantity)                     AS units_sold,
    ROUND(SUM(o.line_revenue), 2)       AS revenue,
    ROUND(SUM(o.quantity * p.margin), 2)AS gross_margin
FROM shopstream.core.silver_orders o
JOIN shopstream.core.silver_products p
ON o.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY p.category;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM shopstream.core.gold_category_performance ORDER BY revenue DESC

category,orders,units_sold,revenue,gross_margin
home-kitchen,1291,1935,284432.61,112616.01
fashion,1052,1556,268969.44,107491.71
books,1058,1533,261403.86,92158.44
fitness,957,1411,258136.76,108812.66
electronics,1152,1674,243726.66,104641.39
toys,915,1326,233182.84,103171.13
grocery,902,1352,220361.17,85506.2
beauty,918,1306,187268.73,71379.89


## 3. Customer lifetime value

In [0]:
%sql
CREATE OR REPLACE TABLE shopstream.core.gold_customer_ltv AS
SELECT
  c.customer_id,
  c.name,
  c.country,
  c.signup_channel,
  COUNT(DISTINCT o.order_id) AS lifetime_orders,
  ROUND(SUM(o.line_revenue), 2) AS lifetime_revenue
FROM shopstream.core.silver_orders o
JOIN shopstream.core.silver_customers c 
USING (customer_id)
WHERE o.status = 'completed'
GROUP BY c.customer_id, c.name, c.country, c.signup_channel;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM shopstream.core.gold_customer_ltv ORDER BY lifetime_revenue DESC LIMIT 10

customer_id,name,country,signup_channel,lifetime_orders,lifetime_revenue
C00088,Mason Brown,US,email,15,6423.06
C00900,Ananya Nair,US,referral,12,6266.62
C00370,James Nair,CA,email,8,5503.26
C00688,Ava Reddy,AE,organic,13,5496.05
C00551,Divya Garcia,IN,paid_search,11,5025.4
C00826,Nikhil Anderson,US,social,10,4978.41
C00424,Yuki Gupta,AU,social,12,4784.18
C00910,Ethan Martinez,CA,email,8,4727.22
C00568,Ethan Lopez,DE,email,8,4684.69
C00230,Ananya Nair,DE,organic,9,4678.66
